# 6.1. Перенос импульса по кольцу

Решаем $u_t+u_x=0$ на $[0,1)$ с периодическими границами.

Точное решение: $u(x,t)=u_0((x-t)\bmod1)$. Точку $x=1$ в сетку не включаем.

Используем две схемы, где $C=\Delta t/\Delta x$:
$$u_j^{n+1}=u_j^n-C(u_j^n-u_{j-1}^n)\qquad\text{(upwind)},$$
$$u_j^{n+1}=u_j^n-\frac C2(u_{j+1}^n-u_{j-1}^n) +\frac{C^2}{2}(u_{j+1}^n-2u_j^n+u_{j-1}^n)\qquad\text{(Лакс–Вендрофф)}.$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import display, Image

methods = ["upwind", "Лакс–Вендрофф"]
sizes = [50, 100, 200]
courants = [0.5, 0.8, 1.0]


def initial(x, kind):
    if kind == "sine":
        return 1 + 0.5*np.sin(2*np.pi*x)
    # Убираем погрешность координат на границах 0.2 и 0.4.
    x = np.round(x % 1, 12)
    return ((x >= 0.2) & (x < 0.4)).astype(float)


def solve(N, C, kind, T=5):
    x = np.arange(N)/N
    u = np.array([initial(x, kind), initial(x, kind)])
    times, history = [0.0], [u.copy()]
    for end in sorted(set([min(1, T), T])):
        while times[-1] < end - 1e-12:
            next_t = min(times[-1] + C/N, end)
            step_c = (next_t - times[-1])*N
            left = np.roll(u, 1, axis=1)
            right = np.roll(u, -1, axis=1)
            u = np.array([
                u[0] - step_c*(u[0] - left[0]),
                u[1] - step_c/2*(right[1] - left[1])
                + step_c**2/2*(right[1] - 2*u[1] + left[1])
            ])
            times.append(next_t)
            history.append(u.copy())
    t, u = np.array(times), np.array(history)
    exact = initial((x[None, :] - t[:, None]) % 1, kind)
    return dict(x=x, t=t, u=u, error=np.mean(np.abs(u-exact[:, None, :]), axis=2), mass=u.mean(axis=2), minimum=u.min(axis=2), maximum=u.max(axis=2))

## Гладкий профиль

Начальное условие $u_0=1+0.5\sin(2\pi x)$, $C=0.8$.
Сравниваем решения после одного и пяти оборотов.

In [ ]:
smooth = {N: solve(N, 0.8, "sine") for N in sizes}
fig, axes = plt.subplots(2, 3, figsize=(11, 6), layout="constrained")
for col, N in enumerate(sizes):
    d = smooth[N]
    for row, T in enumerate([1, 5]):
        k = np.argmin(abs(d["t"] - T))
        ax = axes[row, col]
        ax.plot(d["x"], initial(d["x"]-T, "sine"), "k--", label="точное")
        for m, name in enumerate(methods):
            ax.plot(d["x"], d["u"][k, m], label=name)
        ax.set(title=f"N = {N}, t = {T}", xlabel="x", ylabel="u", ylim=(0.45, 1.55))
axes[0, 0].legend()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), layout="constrained")
for col, T in enumerate([1, 5]):
    for m, name in enumerate(methods):
        errors = [d["error"][np.argmin(abs(d["t"]-T)), m] for d in smooth.values()]
        axes[col].loglog(1/np.array(sizes), errors, "o-", label=name)
    axes[col].set(xlabel="Δx", ylabel="E1", title=f"Ошибка при t = {T}")
    axes[col].legend()
d = smooth[100]
for m, name in enumerate(methods):
    axes[2].plot(d["t"], d["mass"][:, m]-d["mass"][0, m], label=name)
axes[2].set(xlabel="t", ylabel="M(t) - M(0)", title="Изменение количества примеси, N = 100")
axes[2].legend()
plt.show()

Здесь $E_1=\Delta x\sum_j|u_j-u(x_j,t)|$, $M=\Delta x\sum_j u_j$.
Количество примеси сохраняется с точностью округления, но upwind заметно уменьшает амплитуду.
Поэтому сохранение $M$ ещё не означает правильного распределения примеси.
Лакс–Вендрофф на гладком профиле точнее.

## Прямоугольный импульс

Теперь $u_0=1$ при $0.2\le x<0.4$, вне этого интервала — ноль.
Сначала сравним фронты после одного оборота на сетке $N=100$. Отрицательные значения и выбросы не обрезаем.

In [ ]:
box = {(N, C): solve(N, C, "box") for N in sizes for C in courants}
fig, axes = plt.subplots(2, 3, figsize=(11, 6), layout="constrained")
for col, C in enumerate(courants):
    d = box[100, C]
    k = np.argmin(abs(d["t"] - 1))
    for row, center in enumerate([0.2, 0.4]):
        ax = axes[row, col]
        ax.step(d["x"], initial(d["x"], "box"), where="post", color="black", ls="--", label="точное")
        for m, name in enumerate(methods):
            ax.step(d["x"], d["u"][k, m], where="post", label=name)
        ax.set(xlim=(center-0.1, center+0.1), ylim=(-0.25, 1.25), xlabel="x", ylabel="u", title=f"C = {C}, фронт x = {center}")
        
axes[0, 0].legend()
plt.show()

Проверим минимум, максимум и ошибку: сначала во времени при $N=100$, затем при сгущении сетки к моменту $t=5$.

In [ ]:
metrics = [("minimum", "min u"), ("maximum", "max u"), ("error", "E1")]
fig, axes = plt.subplots(2, 3, figsize=(11, 6), layout="constrained")
for m, name in enumerate(methods):
    for col, (key, label) in enumerate(metrics):
        for C in courants:
            d = box[100, C]
            axes[m, col].plot(d["t"], d[key][:, m], label=f"C = {C}")
        axes[m, col].set(xlabel="t", ylabel=label, title=name)
    axes[m, 0].legend()
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(11, 6), layout="constrained")
for m, name in enumerate(methods):
    for col, (key, label) in enumerate(metrics):
        for C in courants:
            values = [box[N, C][key][-1, m] for N in sizes]
            axes[m, col].plot(sizes, values, "o-", label=f"C = {C}")
        axes[m, col].set(xlabel="N", ylabel=label, title=f"{name}, t = 5", xticks=sizes)
    axes[m, 0].legend()
plt.show()

Upwind размывает фронты. Лакс–Вендрофф даёт более резкий переход, но рядом с разрывом
возникают колебания и значения вне $[0,1]$. При сгущении сетки ошибка $E_1$ уменьшается,
но выбросы не обязаны уменьшаться монотонно.

При $C=1$ обе формулы превращаются в $u_j^{n+1}=u_{j-1}^n$: профиль сдвигается ровно на один узел.
Поэтому этот случай даёт точный перенос сеточного профиля (до округления) и не характеризует поведение при других $C$.

## Нарушение условия устойчивости

Отдельно возьмём $C=1.1$, $N=100$ и проследим решение до $t=1$.

In [ ]:
unstable = solve(100, 1.1, "box", T=1)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5), layout="constrained")
for col, (key, label) in enumerate(metrics):
    for m, name in enumerate(methods):
        axes[col].plot(unstable["t"], unstable[key][:, m], label=name)
    axes[col].set_yscale("symlog", linthresh=1)
    axes[col].set(xlabel="t", ylabel=label, title="C = 1.1")
axes[0].legend()
plt.show()

При $C>1$ условие устойчивости нарушено у обеих схем: ошибка и амплитуда колебаний быстро растут.
Для анимации вернёмся к устойчивому расчёту $C=0.8$, $N=100$ и покажем один оборот.

In [ ]:
d = box[100, 0.8]
k1 = np.argmin(abs(d["t"]-1))
frames = np.linspace(0, k1, 81).astype(int)
fig, ax = plt.subplots(figsize=(7, 4))
exact_line, = ax.step(d["x"], initial(d["x"], "box"), where="post", color="black", ls="--", label="точное")
lines = [ax.step(d["x"], d["u"][0, m], where="post", label=name)[0] for m, name in enumerate(methods)]
ax.set(xlim=(0, 1), ylim=(-0.25, 1.25), xlabel="x", ylabel="u", title="C = 0.8, N = 100, t = 0.000")
ax.legend(loc="upper right")
fig.tight_layout()


def update(k):
    t = d["t"][k]
    exact_line.set_ydata(initial((d["x"]-t) % 1, "box"))
    for m, line in enumerate(lines):
        line.set_ydata(d["u"][k, m])
    ax.set_title(f"C = 0.8, N = 100, t = {t:.3f}")


animation = FuncAnimation(fig, update, frames=frames, interval=100)
animation.save("advection_ring.gif", writer=PillowWriter(fps=10), dpi=90)
plt.close(fig)
display(Image(filename="advection_ring.gif"))